# Millers and Traders Survey — Mapping Surveyed Rice Mills

This notebook maps the 798 rice mills surveyed in the 2018 Bangladesh Millers and Traders
Survey (MATS). Exact mill locations are confidential, so mills are **counted in 20 km
hexagons** instead of being plotted as points:

- hexagons with fewer than 3 surveyed mills are not shown;
- median milling capacity is reported only for hexagons with at least 5 mills, rounded to
  2 significant figures;
- names, addresses, interviewer fields and mill IDs are never read.

The rules live in `common.py`; the geometry and drawing code in `millmap.py`.

**Outputs:** `docs/index.html` (interactive map, published with GitHub Pages) and
`figures/*.png` (printable maps).

**Data.** The MATS 2018 microdata are not distributed with this repository — see
`data/README.md`. Point `MATS_DATA_DIR` at the folder that contains `miller_data/`.
The notebook is committed without outputs; `make check` enforces this.

In [ ]:
import pandas as pd

from common import DOCS, FIGURES, HEX_KM, MIN_MILLS, MIN_MILLS_CAPACITY
import millmap

## 1. Load the survey data

Mill location (module A), 2018 mill type and ownership (module C, section 1) and 2018 milling
capacity (module C, sections 4–6). Capacity is the survey's *fortnightly* milling capacity in
metric tons. Ownership is the type of ownership in the first year of each mill's reported
history (variable `c03`), which is not necessarily 2018.

In [ ]:
mills = millmap.load_mills()
print(f'Mills surveyed: {len(mills):,}')
print(mills['mill_type'].value_counts().to_string(), '\n')
print(mills['ownership'].value_counts().to_string(), '\n')
print('Fortnightly milling capacity (MT):')
print(mills['capacity'].describe().round(1).to_string())

## 2. Locate mills and count them per hexagon

Mills are projected to UTM zone 46N and assigned to a regular grid of pointy-top hexagons
20 km across. Mills whose GPS falls outside Bangladesh are dropped. The aggregation checks that
every mill lies inside the hexagon it is counted in. Each hexagon is labelled with the districts
it overlaps, for orientation on the map.

In [ ]:
districts, country = millmap.load_boundaries()
located, excluded = millmap.locate(mills, country)
cells, summary = millmap.aggregate(located)
cells = millmap.add_district_names(cells, districts)
del located, mills   # no row-level data beyond this point

print(f'Excluded: {excluded}')
print(f"{summary['cells']} hexagons of {HEX_KM} km with at least {MIN_MILLS} mills; "
      f"{summary['mills_shown']} of {summary['mills']} mills shown, {summary['mills_hidden']} hidden")
print(f"Capacity reported for {cells['capacity'].notna().sum()} hexagons "
      f"(at least {MIN_MILLS_CAPACITY} mills)")
print(cells['n'].describe().round(1).to_string())

## 3. Interactive map

Written to `docs/index.html`. Use the *Show* menu to switch between all mills, each mill type
and each ownership type, and *Division* to filter. Run `make preview` to view it locally.

In [ ]:
millmap.write_webmap(cells, districts, country, summary, excluded, DOCS / 'index.html')

## 4. Printable maps

In [ ]:
millmap.draw_total(cells, districts, country, summary, excluded, FIGURES / 'rice_mills_hexmap.png')
millmap.draw_by_type(cells, districts, country, summary, excluded, FIGURES / 'rice_mills_hexmap_by_type.png')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIGURES / 'rice_mills_hexmap.png'), width=520))
display(Image(filename=str(FIGURES / 'rice_mills_hexmap_by_type.png'), width=900))